# Phase 3 Debug - Mapping and Coverage Diagnostics

Use this notebook to see how many spiking Phase 2 motor neurons are currently covered by the Phase 3 mapping.

In [ ]:
from pathlib import Path
import sys


def _dedupe_paths(paths):
    seen = set()
    out = []
    for path in paths:
        resolved = path.expanduser().resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        out.append(resolved)
    return out


def _find_phase3_root():
    cwd = Path.cwd().resolve()
    direct = [cwd, *cwd.parents]
    extra = []
    for base in direct:
        extra.extend([base / 'Phase 3_WORKING', base / 'Phase 3'])
    for candidate in _dedupe_paths(direct + extra):
        if (candidate / 'src' / 'phase3_bridge').exists() and (candidate / 'configs' / 'phase3_video_profiles.yaml').exists():
            return candidate
    raise FileNotFoundError('Could not locate the Phase 3 root.')


PHASE3_ROOT = _find_phase3_root()
WORKSPACE_ROOT = next((candidate for candidate in [PHASE3_ROOT.parent, *PHASE3_ROOT.parents] if (candidate / 'Phase 2').exists()), PHASE3_ROOT.parent)
SRC_DIR = PHASE3_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    import pandas as pd
    from phase3_bridge.pipeline import load_phase2_spike_times, load_mapping_csv, summarize_mapping_coverage
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        f"Missing dependency {exc.name!r}. Install the Phase 3 requirements first with: pip install -r {PHASE3_ROOT / 'requirements.txt'}"
    ) from exc

RUN_NAME = 'run_debug_full'
PHASE2_RUN_DIR = WORKSPACE_ROOT / 'Phase 2' / 'data' / 'export_swc' / 'hemi_runs' / RUN_NAME
MAPPING_CSV = (PHASE3_ROOT / 'data' / 'inputs' / 'mappings' / 'mn_to_actuator_mapping_with_groups.csv').resolve()

print('PHASE2_RUN_DIR =', PHASE2_RUN_DIR)
print('MAPPING_CSV    =', MAPPING_CSV)

In [ ]:
spikes = load_phase2_spike_times(PHASE2_RUN_DIR)
mapping = load_mapping_csv(MAPPING_CSV)
coverage = summarize_mapping_coverage(spikes, mapping)

print('[coverage]')
for key, value in coverage.items():
    print(f'  {key}: {value}')

spike_ids = set(spikes['neuron_id'].astype(int).unique())
map_ids = set(mapping['mn_id'].astype(int).unique())

unmapped_spiking = sorted(spike_ids - map_ids)
mapped_not_spiking = sorted(map_ids - spike_ids)

print('\nunmapped spiking ids (first 25):', unmapped_spiking[:25])
print('mapped but not spiking ids (first 25):', mapped_not_spiking[:25])

act_counts = mapping.groupby('actuator_name')['mn_id'].nunique().sort_values(ascending=False)
act_counts.head(30).to_frame('unique_mn_ids')